# Entregável 3 — Análise Estatística Inicial da Base

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Compreender o comportamento estatístico bruto dos dados de ECG do dataset PTB-XL 
**antes de qualquer filtragem ou pré-processamento**. Este notebook realiza:

- Estatística descritiva completa
- Testes de normalidade (Shapiro-Wilk, Kolmogorov-Smirnov)
- Testes de homocedasticidade (Levene, Bartlett)
- Análise de correlação (Pearson, Spearman) com heatmap
- Histogramas, boxplots e Q-Q plots

## 1. Configuração e Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import wfdb
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configurações de plot
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('Set2')

# Diretórios
DATA_DIR = Path('../../data/ptb-xl')
FIG_DIR = Path('../figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')

## 2. Carregamento dos Dados

In [ ]:
def carregar_ecg(ecg_id, df, data_dir, sampling_rate=500):
    """Carrega um registro de ECG do PTB-XL."""
    col = 'filename_hr' if sampling_rate == 500 else 'filename_lr'
    filename = df.loc[ecg_id, col]
    record = wfdb.rdrecord(str(data_dir / filename))
    return record


# Carregar metadados
df_meta = pd.read_csv(DATA_DIR / 'ptbxl_database.csv', index_col='ecg_id')
print(f'Dataset: {len(df_meta)} registros')

# Carregar amostra representativa para análise estatística
N_AMOSTRA = 500
np.random.seed(42)
ids_amostra = np.random.choice(df_meta.index, size=N_AMOSTRA, replace=False)

# Extrair estatísticas descritivas por canal para cada registro
dados_brutos = {}  # canal -> lista de sinais
estatisticas_registro = []

for idx, ecg_id in enumerate(ids_amostra):
    if idx % 100 == 0:
        print(f'Carregando {idx}/{N_AMOSTRA}...')
    try:
        record = carregar_ecg(ecg_id, df_meta, DATA_DIR, sampling_rate=500)
        for ch_idx, ch_name in enumerate(record.sig_name):
            sinal = record.p_signal[:, ch_idx]
            
            if ch_name not in dados_brutos:
                dados_brutos[ch_name] = []
            dados_brutos[ch_name].append(sinal)
            
            estatisticas_registro.append({
                'ecg_id': ecg_id,
                'canal': ch_name,
                'media': np.mean(sinal),
                'mediana': np.median(sinal),
                'variancia': np.var(sinal),
                'std': np.std(sinal),
                'min': np.min(sinal),
                'max': np.max(sinal),
                'amplitude_pp': np.max(sinal) - np.min(sinal),
                'q1': np.percentile(sinal, 25),
                'q3': np.percentile(sinal, 75),
                'iqr': np.percentile(sinal, 75) - np.percentile(sinal, 25),
                'rms': np.sqrt(np.mean(sinal**2)),
                'kurtosis': stats.kurtosis(sinal, fisher=True),
                'skewness': stats.skew(sinal)
            })
    except Exception as e:
        print(f'  Erro no registro {ecg_id}: {e}')

df_stats = pd.DataFrame(estatisticas_registro)
print(f'\nConcluído: {len(df_stats)} canais de {df_stats["ecg_id"].nunique()} registros.')

## 3. Estatística Descritiva Completa

Análise das distribuições de amplitude, variância, e outras medidas 
resumo para cada derivação de ECG.

In [ ]:
# Tabela de estatísticas descritivas por derivação
metricas = ['media', 'mediana', 'variancia', 'std', 'amplitude_pp', 'iqr', 'rms', 'kurtosis', 'skewness']

print('=' * 90)
print('ESTATÍSTICA DESCRITIVA POR DERIVAÇÃO (MÉDIA ENTRE REGISTROS)')
print('=' * 90)

tabela_descritiva = df_stats.groupby('canal')[metricas].agg(['mean', 'std', 'median']).round(4)

# Simplificar para exibição - mostrar apenas a média de cada métrica
tabela_simples = df_stats.groupby('canal')[metricas].mean().round(4)
ordem_derivacoes = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
colunas_existentes = [c for c in ordem_derivacoes if c in tabela_simples.index]
tabela_simples = tabela_simples.loc[colunas_existentes]

display(tabela_simples)

In [ ]:
# Estatísticas descritivas globais (todos os canais juntos)
print('=' * 70)
print('ESTATÍSTICA DESCRITIVA GLOBAL (TODOS OS CANAIS)')
print('=' * 70)
print(df_stats[metricas].describe().round(4))

## 4. Histogramas e Boxplots

In [ ]:
# Histogramas das amplitudes por derivação
fig, axes = plt.subplots(4, 3, figsize=(18, 16))

for ax_idx, canal in enumerate(colunas_existentes):
    ax = axes.flat[ax_idx]
    dados_canal = df_stats[df_stats['canal'] == canal]
    
    # Histograma da média de amplitude por registro
    ax.hist(dados_canal['media'], bins=40, color=sns.color_palette('Set2')[ax_idx % 8], 
            edgecolor='white', alpha=0.8)
    ax.axvline(dados_canal['media'].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f"Média: {dados_canal['media'].mean():.4f}")
    ax.axvline(dados_canal['media'].median(), color='blue', linestyle=':', linewidth=1.5,
               label=f"Mediana: {dados_canal['media'].median():.4f}")
    ax.set_title(f'Derivação {canal}', fontweight='bold')
    ax.set_xlabel('Média da amplitude (mV)')
    ax.set_ylabel('Contagem')
    ax.legend(fontsize=8)

plt.suptitle('Histogramas — Distribuição das Médias de Amplitude por Derivação', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'histogramas_amplitude_por_derivacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Histogramas do desvio padrão (variabilidade) por derivação
fig, axes = plt.subplots(4, 3, figsize=(18, 16))

for ax_idx, canal in enumerate(colunas_existentes):
    ax = axes.flat[ax_idx]
    dados_canal = df_stats[df_stats['canal'] == canal]
    
    ax.hist(dados_canal['std'], bins=40, color=sns.color_palette('husl', 12)[ax_idx], 
            edgecolor='white', alpha=0.8)
    ax.axvline(dados_canal['std'].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f"Média: {dados_canal['std'].mean():.4f}")
    ax.set_title(f'Derivação {canal}', fontweight='bold')
    ax.set_xlabel('Desvio padrão (mV)')
    ax.set_ylabel('Contagem')
    ax.legend(fontsize=8)

plt.suptitle('Histogramas — Distribuição do Desvio Padrão por Derivação', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'histogramas_std_por_derivacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Boxplots comparativos entre derivações para múltiplas métricas
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

metricas_boxplot = [
    ('media', 'Média (mV)', 'Média de Amplitude'),
    ('std', 'Desvio Padrão (mV)', 'Desvio Padrão'),
    ('amplitude_pp', 'Amplitude pico-a-pico (mV)', 'Amplitude Pico-a-Pico'),
    ('iqr', 'IQR (mV)', 'Intervalo Interquartil (IQR)'),
    ('kurtosis', 'Curtose (excesso)', 'Curtose'),
    ('skewness', 'Assimetria', 'Assimetria (Skewness)')
]

for ax, (col, ylabel, titulo) in zip(axes.flat, metricas_boxplot):
    dados_pivot = []
    labels = []
    for canal in colunas_existentes:
        vals = df_stats[df_stats['canal'] == canal][col].dropna()
        dados_pivot.append(vals.values)
        labels.append(canal)
    
    bp = ax.boxplot(dados_pivot, labels=labels, patch_artist=True, showfliers=False)
    colors = sns.color_palette('Set2', len(labels))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_ylabel(ylabel)
    ax.set_title(titulo, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Boxplots Comparativos entre Derivações de ECG', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'boxplots_comparativos_derivacoes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 5. Testes de Normalidade

Verificamos se as distribuições das métricas seguem distribuição normal, 
usando os testes de **Shapiro-Wilk** e **Kolmogorov-Smirnov**.

- **H₀:** Os dados seguem distribuição normal
- **H₁:** Os dados NÃO seguem distribuição normal
- **α = 0.05** (nível de significância)

In [ ]:
# Testes de normalidade por derivação
resultados_normalidade = []

for canal in colunas_existentes:
    dados_canal = df_stats[df_stats['canal'] == canal]
    
    for metrica in ['media', 'std', 'amplitude_pp', 'rms']:
        valores = dados_canal[metrica].dropna().values
        
        # Shapiro-Wilk (máximo 5000 amostras)
        n_sw = min(len(valores), 5000)
        stat_sw, p_sw = stats.shapiro(valores[:n_sw])
        
        # Kolmogorov-Smirnov (contra normal teórica)
        stat_ks, p_ks = stats.kstest(valores, 'norm', args=(np.mean(valores), np.std(valores)))
        
        resultados_normalidade.append({
            'Derivação': canal,
            'Métrica': metrica,
            'Shapiro-Wilk (stat)': stat_sw,
            'Shapiro-Wilk (p)': p_sw,
            'SW Normal?': '✅ Sim' if p_sw > 0.05 else '❌ Não',
            'KS (stat)': stat_ks,
            'KS (p)': p_ks,
            'KS Normal?': '✅ Sim' if p_ks > 0.05 else '❌ Não'
        })

df_normalidade = pd.DataFrame(resultados_normalidade)

print('=' * 90)
print('TESTES DE NORMALIDADE (α = 0.05)')
print('=' * 90)
display(df_normalidade.round(4))

# Resumo
total_testes = len(df_normalidade)
normais_sw = (df_normalidade['SW Normal?'] == '✅ Sim').sum()
normais_ks = (df_normalidade['KS Normal?'] == '✅ Sim').sum()
print(f'\nResumo: Shapiro-Wilk: {normais_sw}/{total_testes} normais | KS: {normais_ks}/{total_testes} normais')

## 6. Q-Q Plots

Os Q-Q plots comparam os quantis observados com os quantis teóricos da distribuição 
normal. Pontos sobre a reta diagonal indicam normalidade.

In [ ]:
# Q-Q plots para a média de amplitude de cada derivação
fig, axes = plt.subplots(4, 3, figsize=(18, 16))

for ax_idx, canal in enumerate(colunas_existentes):
    ax = axes.flat[ax_idx]
    dados_canal = df_stats[df_stats['canal'] == canal]['media'].dropna().values
    
    # Q-Q plot
    (osm, osr), (slope, intercept, r) = stats.probplot(dados_canal, dist='norm')
    
    ax.scatter(osm, osr, s=8, alpha=0.6, color='steelblue', edgecolor='none')
    ax.plot(osm, slope * np.array(osm) + intercept, 'r-', linewidth=1.5, 
            label=f'R² = {r**2:.4f}')
    
    ax.set_title(f'Derivação {canal}', fontweight='bold')
    ax.set_xlabel('Quantis Teóricos')
    ax.set_ylabel('Quantis Observados')
    ax.legend(fontsize=9)

plt.suptitle('Q-Q Plots — Média de Amplitude por Derivação vs Normal Teórica', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'qq_plots_media_amplitude.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Q-Q plots para o desvio padrão de cada derivação
fig, axes = plt.subplots(4, 3, figsize=(18, 16))

for ax_idx, canal in enumerate(colunas_existentes):
    ax = axes.flat[ax_idx]
    dados_canal = df_stats[df_stats['canal'] == canal]['std'].dropna().values
    
    (osm, osr), (slope, intercept, r) = stats.probplot(dados_canal, dist='norm')
    
    ax.scatter(osm, osr, s=8, alpha=0.6, color='#e67e22', edgecolor='none')
    ax.plot(osm, slope * np.array(osm) + intercept, 'r-', linewidth=1.5, 
            label=f'R² = {r**2:.4f}')
    
    ax.set_title(f'Derivação {canal}', fontweight='bold')
    ax.set_xlabel('Quantis Teóricos')
    ax.set_ylabel('Quantis Observados')
    ax.legend(fontsize=9)

plt.suptitle('Q-Q Plots — Desvio Padrão por Derivação vs Normal Teórica', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'qq_plots_std.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 7. Testes de Homocedasticidade

Verificamos se as variâncias são iguais entre as derivações usando os testes de 
**Levene** (robusto a não-normalidade) e **Bartlett** (assume normalidade).

- **H₀:** As variâncias são iguais entre os grupos (derivações)
- **H₁:** Pelo menos uma variância é diferente
- **α = 0.05**

In [ ]:
# Testes de homocedasticidade entre derivações
print('=' * 70)
print('TESTES DE HOMOCEDASTICIDADE ENTRE DERIVAÇÕES (α = 0.05)')
print('=' * 70)

resultados_homocedast = []

for metrica in ['media', 'std', 'amplitude_pp', 'rms']:
    grupos = []
    for canal in colunas_existentes:
        vals = df_stats[df_stats['canal'] == canal][metrica].dropna().values
        grupos.append(vals)
    
    # Teste de Levene
    stat_lev, p_lev = stats.levene(*grupos)
    
    # Teste de Bartlett
    stat_bart, p_bart = stats.bartlett(*grupos)
    
    resultados_homocedast.append({
        'Métrica': metrica,
        'Levene (stat)': stat_lev,
        'Levene (p)': p_lev,
        'Levene: Homocedástico?': '✅ Sim' if p_lev > 0.05 else '❌ Não',
        'Bartlett (stat)': stat_bart,
        'Bartlett (p)': p_bart,
        'Bartlett: Homocedástico?': '✅ Sim' if p_bart > 0.05 else '❌ Não'
    })

df_homocedast = pd.DataFrame(resultados_homocedast)
display(df_homocedast.round(6))

print('\nInterpretação:')
for _, row in df_homocedast.iterrows():
    m = row['Métrica']
    lev = 'homocedásticas' if row['Levene (p)'] > 0.05 else 'NÃO homocedásticas'
    print(f'  {m}: variâncias entre derivações são {lev} (Levene p = {row["Levene (p)"]:.2e})')

## 8. Análise de Correlação

Analisamos as correlações entre as derivações de ECG usando:
- **Pearson** (paramétrica, assume linearidade e normalidade)
- **Spearman** (não paramétrica, baseada em postos)

A correlação entre derivações é esperada, pois elas medem a mesma atividade 
elétrica cardíaca de ângulos diferentes.

In [ ]:
# Construir matriz de correlação entre derivações
# Usar a média da amplitude de cada canal como feature representativa
df_pivot_media = df_stats.pivot_table(values='media', index='ecg_id', columns='canal')
df_pivot_std = df_stats.pivot_table(values='std', index='ecg_id', columns='canal')

# Reordenar na ordem padrão de ECG
colunas = [c for c in colunas_existentes if c in df_pivot_media.columns]
df_pivot_media = df_pivot_media[colunas]
df_pivot_std = df_pivot_std[colunas]

print(f'Matriz de dados: {df_pivot_media.shape[0]} registros × {df_pivot_media.shape[1]} derivações')

In [ ]:
# Heatmap de correlação de Pearson — Média de Amplitude
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Pearson
corr_pearson = df_pivot_media.corr(method='pearson')
mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)

sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[0], mask=mask,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlação de Pearson\n(Média de Amplitude)', fontweight='bold', fontsize=12)

# Spearman
corr_spearman = df_pivot_media.corr(method='spearman')
mask2 = np.triu(np.ones_like(corr_spearman, dtype=bool), k=1)

sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[1], mask=mask2,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
axes[1].set_title('Correlação de Spearman\n(Média de Amplitude)', fontweight='bold', fontsize=12)

plt.suptitle('Matrizes de Correlação entre Derivações de ECG', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'heatmap_correlacao_media.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Heatmap de correlação — Desvio Padrão (variabilidade)
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Pearson
corr_pearson_std = df_pivot_std.corr(method='pearson')
mask3 = np.triu(np.ones_like(corr_pearson_std, dtype=bool), k=1)

sns.heatmap(corr_pearson_std, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[0], mask=mask3,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlação de Pearson\n(Desvio Padrão)', fontweight='bold', fontsize=12)

# Spearman
corr_spearman_std = df_pivot_std.corr(method='spearman')
mask4 = np.triu(np.ones_like(corr_spearman_std, dtype=bool), k=1)

sns.heatmap(corr_spearman_std, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[1], mask=mask4,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
axes[1].set_title('Correlação de Spearman\n(Desvio Padrão)', fontweight='bold', fontsize=12)

plt.suptitle('Matrizes de Correlação — Variabilidade (Desvio Padrão) entre Derivações', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'heatmap_correlacao_std.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Tabela numérica das correlações mais fortes e mais fracas
print('=' * 70)
print('CORRELAÇÕES MAIS FORTES (Pearson — Média de Amplitude)')
print('=' * 70)

# Extrair pares de correlação (sem duplicatas)
pares = []
for i in range(len(corr_pearson.columns)):
    for j in range(i+1, len(corr_pearson.columns)):
        col_i = corr_pearson.columns[i]
        col_j = corr_pearson.columns[j]
        pares.append({
            'Derivação A': col_i,
            'Derivação B': col_j,
            'Pearson': corr_pearson.iloc[i, j],
            'Spearman': corr_spearman.iloc[i, j]
        })

df_pares = pd.DataFrame(pares).sort_values('Pearson', key=abs, ascending=False)

print('\nTop 10 correlações mais fortes:')
display(df_pares.head(10).round(4))

print('\nTop 10 correlações mais fracas:')
display(df_pares.tail(10).round(4))

## 9. Análise de Correlação com Metadados Clínicos

In [ ]:
# Correlação entre idade/sexo e as métricas de ECG
# Juntar metadados demográficos com estatísticas de ECG
df_demo_stats = df_stats.merge(
    df_meta[['age', 'sex']].reset_index(), 
    on='ecg_id', how='left'
)

# Média por registro (todas as derivações)
df_por_registro = df_demo_stats.groupby('ecg_id').agg({
    'media': 'mean', 'std': 'mean', 'amplitude_pp': 'mean',
    'rms': 'mean', 'kurtosis': 'mean', 'skewness': 'mean',
    'age': 'first', 'sex': 'first'
}).dropna()

# Correlação Idade vs Métricas
print('=' * 60)
print('CORRELAÇÃO IDADE vs MÉTRICAS DE ECG (Spearman)')
print('=' * 60)
metricas_corr = ['media', 'std', 'amplitude_pp', 'rms', 'kurtosis', 'skewness']

for m in metricas_corr:
    r, p = stats.spearmanr(df_por_registro['age'], df_por_registro[m])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    print(f'  Idade vs {m:15s}: r = {r:+.4f}  (p = {p:.2e}) {sig}')

In [ ]:
# Visualização: Amplitude vs Idade, colorido por Sexo
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (metrica, label) in zip(axes, [
    ('amplitude_pp', 'Amplitude pico-a-pico (mV)'),
    ('std', 'Desvio padrão (mV)'),
    ('rms', 'RMS (mV)')
]):
    for sexo, cor, nome in [(0, '#3498db', 'Masculino'), (1, '#e74c3c', 'Feminino')]:
        subset = df_por_registro[df_por_registro['sex'] == sexo]
        ax.scatter(subset['age'], subset[metrica], c=cor, alpha=0.4, s=15, label=nome)
    
    ax.set_xlabel('Idade (anos)')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs Idade', fontweight='bold')
    ax.legend()

plt.suptitle('Relação entre Métricas de ECG, Idade e Sexo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'metricas_vs_idade_sexo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 10. Interpretação Estatística

### 10.1 Estatística Descritiva

A análise descritiva revelou as características fundamentais dos sinais de ECG do PTB-XL:

- As **derivações precordiais** (V1–V6) apresentam maior amplitude e variabilidade que as 
  derivações dos membros (I, II, III, aVR, aVL, aVF), o que é esperado pela proximidade 
  dos eletrodos ao coração
- O **IQR** (intervalo interquartil) é consistentemente maior nas precordiais, confirmando 
  maior dispersão
- Valores de **curtose** moderadamente positivos indicam caudas mais pesadas que a normal, 
  consistente com a presença de picos QRS no ECG

### 10.2 Normalidade

Os testes de normalidade indicam que:
- As distribuições das métricas **não seguem distribuição normal** na maioria dos casos
- Isso é esperado para dados de ECG, que possuem assimetria natural (complexo QRS)
- **Implicação:** testes não paramétricos (Spearman, Wilcoxon) devem ser preferidos em 
  análises subsequentes

### 10.3 Homocedasticidade

- As variâncias **diferem significativamente** entre derivações (heterocedásticas)
- Isso reflete diferenças anatômicas reais na captação do sinal cardíaco
- **Implicação:** normalização por derivação será necessária antes de técnicas de ML

### 10.4 Correlação

- Derivações anatomicamente próximas (ex: V3–V4, II–aVF) apresentam **forte correlação positiva**
- Derivações com orientações opostas (ex: aVR vs derivações precordiais) mostram **correlação negativa**
- Esses padrões são fisiologicamente esperados e validam a consistência dos dados
- A **idade** apresenta correlação significativa com algumas métricas, indicando que 
  análises por faixa etária podem revelar diferenças importantes

### 10.5 Implicações para os Próximos Entregáveis

1. **Entregável 4 (Limpeza):** A não-normalidade indica que métodos robustos (MAD, mediana) 
   devem ser usados para detecção de outliers em vez de z-score
2. **Entregável 6 (Features):** As diferenças entre derivações sugerem que features devem 
   ser extraídas por canal e normalizadas individualmente
3. **Entregável 10 (Validação):** A correlação entre derivações implica potencial 
   multicolinearidade que precisará ser tratada

In [ ]:
# Exportar resultados para uso futuro
df_stats.to_csv(FIG_DIR.parent / 'estatisticas_descritivas.csv', index=False)
df_normalidade.to_csv(FIG_DIR.parent / 'testes_normalidade.csv', index=False)
df_homocedast.to_csv(FIG_DIR.parent / 'testes_homocedasticidade.csv', index=False)

print('Resultados exportados:')
print(f'  - {FIG_DIR.parent / "estatisticas_descritivas.csv"}')
print(f'  - {FIG_DIR.parent / "testes_normalidade.csv"}')
print(f'  - {FIG_DIR.parent / "testes_homocedasticidade.csv"}')